# CR-1a G3 分析と監査（CPU・API なし）

凍結した計画の G3 です。G1 の土台の道と G2 の結果ファイル（どちらも Drive の非公開ファイル）だけから計算します。

- **先に確かめること**：G2 の記録のハッシュ、23 個の結果ファイルのハッシュ、計画したリクエストがちょうど全部そろっていること、応答のモデル、土台の道のハッシュ。どれかが違えば止まります。
- **監査**：切れ目の 5% を、別に書いた参照の実装で計算し直してずれが 0 か、陰性対照が同じ乱数種で同じ p 値になるか、など（計画の G3 のとおり）。監査に落ちたら、ラベルは出しません。
- **分析**：分析に入る条件とラベルの優先順 → 主0〜主3、確かめ C1〜C4、副次 S2・S4・S5・S6・S8、記録 S7。
- 問題文・応答はどこにも表示しません。結果は数・ラベル・番号だけで、Drive の `DECISION_POINTS/cr1a/G3/CR1A_G3_result.json` に入ります。
- 費用 $0。シークレットは `GITHUB_TOKEN` だけ。セル1 → セル2（5〜15 分くらい）。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact head, notifications.  No API key; the dataset is read without any token.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
STUDY = "CR-1a"
EXPECTED_HEAD = "bcafae783d2a369f3e8363434b40c981b3490c6f"
G1_HEAD = "7d95ee90a7b536fb85326e2a30981b9bc8030421"
G2_HEAD = "f6c6bb94fe2ff77a5726e71b835f7ee0632a9d4d"
G1_ITEMS_SHA = "bd42639245cb234f86e0bcd97092db67ae2842c9f1e48e123a9ee1b484c931b5"
G2_SUMMARY_SHA = "17afa043db733289e49eb4d55ac1c160760067f36eb2a9c4cdfa36e956f159f1"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
for anc in (G1_HEAD, G2_HEAD, "69b43d8", "4dcae3f"):
    assert subprocess.run(["git", "-C", str(REPO), "merge-base", "--is-ancestor", anc, head]).returncode == 0, anc + " が祖先ではありません"
token = None
env.pop("GITHUB_TOKEN", None)
sys.path.insert(0, str(REPO))
from pork import cr1_run as R
from pork import cr1_requests as RQ
from pork import cr1_analysis as A
from pork import cr1_g3 as G3
import numpy
print("exact head:", head, "| numpy", numpy.__version__, "| python", sys.version.split()[0])

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'CR-1a G3 分析と監査（CPU・API なし）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - CR-1a G3 analysis', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — integrity checks, audit, analysis; numbers / labels / ids only.
t0 = time.time()
from pork import cr1_items as CI
rows, DATA_ID = R.load_rows_public()
chain = CI.chain_and_candidates(rows)
del rows
assert CI.candidates_sha256(chain) == R.CANDIDATES_SHA256, "候補の一覧が段2 と違います。止めます。"
D = Path("/content/drive/MyDrive/DECISION_POINTS/cr1a")
ck = json.loads((D / "G1/CR1A_G1_checkpoint_PRIVATE.json").read_text())
assert ck["header"] == R.header(STUDY, G1_HEAD, R.CANDIDATES_SHA256, A.MODEL, RQ.SDK_VERSION)
R.validate(ck, chain)
g1 = R.summarize_g1(ck)
assert g1["verdict"] == "GO" and g1["final_items_ids_sha256"] == G1_ITEMS_SHA
s2_text = (D / "G2/CR1A_G2_summary.json").read_text()
assert hashlib.sha256(s2_text.encode("utf-8")).hexdigest() == G2_SUMMARY_SHA, "G2 の記録が変わっています。止めます。"
g2 = json.loads(s2_text)
assert g2["done"] and g2["code_head"] == G2_HEAD and g2["requests"] == sum((fi["L"] + 1) * A.K for fi in g1["final_items"])
res, load_stats = G3.load_results(D / "G2", g2["batch_records"], g1["final_items"])
print("読み込み:", load_stats["requests"], "件 | 状態:", load_stats["final_status"], "| モデル:", list(load_stats["models"]))
out = G3.run(g1["final_items"], ck["items"], res, unresolved_requests=g2["unresolved"])
out["S7"] = G3.s7(ck, g1, g2, out["_items"])
out["load"] = load_stats
out["run"] = {"code_head": head, "g1_code_head": G1_HEAD, "g2_code_head": G2_HEAD, "dataset": DATA_ID,
              "numpy": numpy.__version__, "python": sys.version.split()[0], "seconds": round(time.time() - t0, 1)}
OUT = D / "G3"
OUT.mkdir(parents=True, exist_ok=True)
text = G3.to_json(out)
R._atomic_write(OUT / "CR1A_G3_result.json", json.loads(text))
print("監査:", "合格" if out["audit"]["pass"] else "不合格", "| 判定:", out["verdict"], "| 時間:", out["run"]["seconds"], "秒")
notify('終わりました：CR-1a の G3（分析と監査）の結果を Drive に書きました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()